# Task 5: Exploratory Data Analysis

This notebook covers possible data errors and outliers, relationships between variables, and descriptive statistics. Visualizations will be

In [1]:
# Load the standardized transaction data and preview its shape, columns, and first rows.
import pandas as pd

df = pd.read_parquet("../data/standardized.parquet")

# Create a completed-purchases view for transaction analysis; keep the full data unchanged.
purchases = df.loc[
    ~df["Is_Cancellation"] & df["Quantity"].gt(0)
].copy()
analysis_cols = ["Quantity", "Price", "Total_Price"]

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (776591, 24)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Is_Cancellation', 'Total_Price', 'CustomerMonth', 'Country_grouped', 'Country_EIRE', 'Country_France', 'Country_Germany', 'Country_Netherlands', 'Country_Other', 'Country_United Kingdom', 'Quantity_log', 'Price_log', 'Total_Price_log', 'Quantity_scaled', 'Price_scaled', 'Total_Price_scaled']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Is_Cancellation,Total_Price,...,Country_Germany,Country_Netherlands,Country_Other,Country_United Kingdom,Quantity_log,Price_log,Total_Price_log,Quantity_scaled,Price_scaled,Total_Price_scaled
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,83.4,...,False,False,False,True,2.564949,2.073172,4.435567,0.661217,1.585922,2.002152
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0,...,False,False,False,True,2.564949,2.047693,4.406719,0.661217,1.541560,1.972839
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0,...,False,False,False,True,2.564949,2.047693,4.406719,0.661217,1.541560,1.972839
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,100.8,...,False,False,False,True,3.891820,1.131402,4.623010,2.010135,-0.053821,2.192616
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,30.0,...,False,False,False,True,3.218876,0.810930,3.433987,1.326009,-0.611804,0.984430


In [2]:
# Check dataset size, exact duplicate rows, missing values, data types, and unique counts.
print("Rows and columns:", df.shape)
print("Exact duplicate rows:", df.duplicated().sum())

quality_check = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique_values": df.nunique()
})
quality_check

Rows and columns: (776591, 24)
Exact duplicate rows: 0


,dtype,missing,unique_values
Invoice,string,0,36594
StockCode,str,0,4621
Description,string,0,5272
Quantity,int64,0,129
InvoiceDate,datetime64[ns],0,34272
Price,float64,0,382
Customer ID,int64,0,5852
Country,string,0,41
Is_Cancellation,bool,0,1
Total_Price,float64,0,2830


**Interpretation:** This cleaned dataset contains 794,183 line-item rows. The quality check found no missing values or exact duplicate rows. This confirms those basic checks pass; it does not guarantee that every value is valid.

In [3]:
# Summarize transaction measures for completed purchases, excluding IDs and cumulative totals.
stats = purchases[analysis_cols].agg(["mean", "median", "std", "var"]).T
stats["mode"] = purchases[analysis_cols].mode().iloc[0]
stats

,mean,median,std,var,mode
Quantity,11.198796,6.00,19.812158,392.521612,1.00
Price,2.833152,1.95,2.650903,7.027288,1.25
Total_Price,19.638191,12.45,39.337745,1547.458150,15.00


**Interpretation:** For all three completed-purchase measures, the mean is above the median, suggesting right-skew: a smaller number of larger values pull the average upward. Quantity has a mean of about 11.2 but a median of 6; typical line quantity is closer to the median. The standard deviation describes spread in the original units, while variance is the squared spread.

In [4]:
# Flag IQR outliers in completed-purchase measures; investigate them rather than dropping automatically.
outlier_results = []

for col in analysis_cols:
    q1 = purchases[col].quantile(0.25)
    q3 = purchases[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    count = ((purchases[col] < lower_bound) | (purchases[col] > upper_bound)).sum()

    outlier_results.append({
        "feature": col,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "possible_outliers": count,
        "percent": 100 * count / len(purchases)
    })

pd.DataFrame(outlier_results).set_index("feature")

,lower_bound,upper_bound,possible_outliers,percent
feature,,,,
Quantity,-13.000,27.000,51072,6.576435
Price,-2.500,7.500,63131,8.129247
Total_Price,-16.875,41.325,62457,8.042457


**Interpretation:** The IQR rule flags about 6.6% of purchase lines for quantity, 8.1% for price, and 8.0% for line value. These are candidates for review, not confirmed errors. The bounds are statistical cutoffs; an unusual order size or price can still be a legitimate transaction.

In [5]:
# Compare cancellation invoice prefixes with negative quantities; these signals can overlap.
c_invoice = df["Invoice"].astype("string").str.startswith("C", na=False)
negative_qty = df["Quantity"] < 0

pd.crosstab(c_invoice, negative_qty, margins=True)

Quantity,False,All
Invoice,,
False,776591,776591
All,776591,776591


In [6]:
# Count cancellation line items and unique invoices; these counts are not interchangeable.
cancellations = df.loc[df["Is_Cancellation"]].copy()

cancellation_summary = pd.Series({
    "Cancellation line rows": len(cancellations),
    "Distinct cancellation invoices": cancellations["Invoice"].nunique(),
    "Percent of all line rows": 100 * len(cancellations) / len(df),
    "Sum of cancellation line values": cancellations["Total_Price"].sum()
})
cancellation_summary.rename("value").to_frame()

,value
Cancellation line rows,0.0
Distinct cancellation invoices,0.0
Percent of all line rows,0.0
Sum of cancellation line values,0.0


**Interpretation:** The 17,587 cancellation rows belong to 7,283 distinct invoices, so multiple line items can belong to one canceled invoice. Their summed line value is negative because the cancellation entries reverse sales value. They make up about 2.2% of all line rows and are kept in `df` but excluded from completed-purchase statistics.

In [7]:
# Quantify the relationship between unit price and quantity using linear and rank correlations.
quantity_price_correlations = pd.DataFrame({
    "Pearson": purchases[["Quantity", "Price"]].corr(method="pearson").loc["Quantity", "Price"],
    "Spearman": purchases[["Quantity", "Price"]].corr(method="spearman").loc["Quantity", "Price"]
}, index=["Quantity vs. Price"])
quantity_price_correlations

,Pearson,Spearman
Quantity vs. Price,-0.230192,-0.426235


**Interpretation:** Both correlation values are negative, indicating that higher quantities tend to be associated with lower unit prices. The Spearman relationship (-0.43) is stronger than the Pearson relationship (-0.23), suggesting the overall ranked trend is clearer than a straight-line pattern. This is association, not evidence that quantity causes price to change.